# L03 — Enzyme Kinetics I

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lab-biotek-bio-ugm/TKBM262615_Practicals/blob/main/notebooks/03_enzyme_kinetics_I.ipynb)

**Course:** TBM263214 Komputasi Biologi (Computational Biology) · **Week 3** · **CPMK2**

Companion notebook for [`scripts/03_enzyme_kinetics_I.py`](../scripts/03_enzyme_kinetics_I.py), based on the course lecture note [`03-enzyme-kinetics-1-en.md`](https://github.com/lab-biotek-bio-ugm/TKBM262615/blob/main/lecture-notes/03-enzyme-kinetics-1-en.md).

Simulates the enzyme mechanism E + S ⇌ ES → E + P, checks the quasi-steady-state assumption (QSSA) for [ES] against the full mass-action model, and derives the Michaelis-Menten rate law.


## Learning Objectives

After this notebook, you should be able to:
1. **Write the mass-action ODEs** for the enzyme mechanism E + S ⇌ ES → E + P. *(CPMK2)*
2. **State and apply the quasi-steady-state assumption (QSSA)** for the enzyme-substrate complex [ES]. *(CPMK2)*
3. **Derive the Michaelis-Menten equation** v = Vmax[S] / (KM + [S]) from the QSSA. *(CPMK2)*
4. **Interpret Vmax and KM**, and locate them on a simulated v0(S) curve. *(CPMK2)*


In [ ]:
# Setup — Colab already ships numpy, scipy, and matplotlib, so no pip install is needed.
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


## Key Concepts

- **Enzyme mechanism:** E + S ⇌ ES → E + P, with forward binding $k_1$, unbinding $k_{-1}$, and catalytic turnover $k_{cat}$.
- **Total-enzyme conservation:** free enzyme + bound enzyme is constant, $[E] + [ES] = E_{tot}$.
- **Quasi-steady-state assumption (QSSA):** when $E_{tot} \ll [S]_0$, $[ES]$ rises to a plateau almost instantly and then tracks $[S]$ adiabatically, so $\frac{d[ES]}{dt} \approx 0$.
- **Michaelis constant $K_M$:** the substrate concentration at which $v_0 = V_{max}/2$; a lower $K_M$ means tighter substrate binding.
- **$V_{max}$:** the maximum rate, reached when the enzyme is saturated with substrate ($[S] \gg K_M$).

## Key Equations

- Full mass-action system:
$$ \frac{d[ES]}{dt} = k_1[E][S] - k_{-1}[ES] - k_{cat}[ES], \qquad \frac{d[P]}{dt} = k_{cat}[ES] $$

- QSSA ($\frac{d[ES]}{dt}=0$) with $[E]=E_{tot}-[ES]$ gives:
$$ [ES] = \frac{E_{tot}[S]}{K_M + [S]}, \qquad K_M = \frac{k_{-1}+k_{cat}}{k_1} $$

- Michaelis-Menten rate law:
$$ v = k_{cat}[ES] = \frac{V_{max}[S]}{K_M + [S]}, \qquad V_{max} = k_{cat} E_{tot} $$


In [ ]:
# Full mechanism: E + S <-> ES -> E + P
def enzyme_full(t, y, k1, km1, kcat):
    E, S, ES, P = y
    v_f = k1 * E * S
    v_r = km1 * ES
    v_cat = kcat * ES
    dE = -v_f + v_r + v_cat
    dS = -v_f + v_r
    dES = v_f - v_r - v_cat
    dP = v_cat
    return [dE, dS, dES, dP]

k1, km1, kcat = 100.0, 50.0, 10.0   # 1/(uM s), 1/s, 1/s
Etot = 1.0                          # uM
KM = (km1 + kcat) / k1
Vmax = kcat * Etot
print(f"KM = {KM:.3f} uM, Vmax = {Vmax:.3f} uM/s")


In [ ]:
# Progress curve: S0 >> Etot, so the QSSA should hold
S0 = 100.0
t_eval = np.linspace(0, 15, 600)
sol = solve_ivp(enzyme_full, (0, 15), [Etot, S0, 0.0, 0.0],
                 t_eval=t_eval, args=(k1, km1, kcat), rtol=1e-8, atol=1e-10)
E, S, ES, P = sol.y

# Compare simulated [ES] to the QSSA approximation ES = Etot*S/(KM+S).
# The QSSA only holds after the initial fast transient (timescale ~1/(k1*S0+k-1+kcat)),
# during which [ES] is still rising from 0 toward its plateau.
ES_qssa = Etot * S / (KM + S)
t_fast = 5.0 / (k1 * S0 + km1 + kcat)
after_transient = sol.t > 10 * t_fast
max_err_all = np.max(np.abs(ES - ES_qssa)) / Etot
max_err_after = np.max(np.abs(ES[after_transient] - ES_qssa[after_transient])) / Etot
print(f"QSSA check (all t, including the initial fast transient): max relative error = {max_err_all:.4f}")
print(f"QSSA check (t > {10 * t_fast:.4f} s, after the fast transient): max relative error = {max_err_after:.4f}")


In [ ]:
# Initial rate v0(S0): hold [S] fixed, integrate the (E, ES) subsystem to
# steady state, then v0 = kcat*[ES]_ss. Compared to the analytic MM curve.
def enzyme_fixed_S(t, y, k1, km1, kcat, S_val):
    E, ES = y
    v_f = k1 * E * S_val
    v_r = km1 * ES
    v_cat = kcat * ES
    return [-v_f + v_r + v_cat, v_f - v_r - v_cat]

def initial_rate(S0_val, t_ss=1.0):
    s = solve_ivp(enzyme_fixed_S, (0, t_ss), [Etot, 0.0],
                   args=(k1, km1, kcat, S0_val), rtol=1e-10, atol=1e-14)
    return kcat * s.y[1, -1]

S_values = np.logspace(-2, 2, 25)
v_sim = np.array([initial_rate(s) for s in S_values])
v_analytic = Vmax * S_values / (KM + S_values)
print(f"At S = KM: v_sim = {initial_rate(KM):.3f} uM/s, Vmax/2 = {Vmax / 2:.3f} uM/s")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(sol.t, S, label="S")
axes[0].plot(sol.t, P, label="P")
axes[0].set_title("Progress curve S(t), P(t)")
axes[0].legend()

axes[1].plot(sol.t, ES, label="[ES] full")
axes[1].plot(sol.t, ES_qssa, "--", label="[ES] QSSA")
axes[1].set_title("[ES]: full simulation vs QSSA")
axes[1].legend()

axes[2].semilogx(S_values, v_sim, "o", label="v0 simulated")
axes[2].semilogx(S_values, v_analytic, "-", label="Vmax S / (KM + S)")
axes[2].axhline(Vmax / 2, color="gray", ls=":", lw=1)
axes[2].axvline(KM, color="gray", ls=":", lw=1)
axes[2].set_title("Michaelis-Menten curve v0(S)")
axes[2].legend()

for ax in axes:
    ax.set_xlabel("time t" if ax is not axes[2] else "[S]")
    ax.grid(alpha=0.3)
axes[2].set_ylabel("v0")
plt.tight_layout()
plt.show()


## Python Exercises

### Exercise 1 *(CPMK2)*
Re-run the QSSA check with $E_{tot} = 20$ (so $E_{tot}$ is no longer $\ll S_0$). How does `max_err_after` (the post-transient error) change, and why?


In [ ]:
# TODO: Exercise 1 — rerun the QSSA comparison with Etot = 20, report the new max_err_after and explain the change


### Exercise 2 *(CPMK2)*
From the simulated `S_values`/`v_sim` arrays, find the value of `S` closest to $K_M$ and confirm that `v_sim` there is close to $V_{max}/2$.


In [ ]:
# TODO: Exercise 2 — find the S_values entry closest to KM and compare its v_sim to Vmax/2


### Exercise 3 *(CPMK2)*
Halve $k_{cat}$ (keep $k_1, k_{-1}$ fixed) and recompute $K_M$ and $V_{max}$. Which one changes, and which stays the same?


In [ ]:
# TODO: Exercise 3 — recompute KM and Vmax for kcat/2, compare to the original values


## Discussion Questions

1. Why does the QSSA break down when $E_{tot}$ is not much smaller than $[S]_0$?
2. What does a very small $K_M$ imply biologically about an enzyme's affinity for its substrate?
3. The progress curve looks almost linear at the start when $S_0 \gg K_M$ — why?

## Reading

- Ingalls (2013) Chapter 3: enzyme kinetics, the QSSA, and the derivation of the Michaelis-Menten equation.
- Alon (2006) Chapter 2: Michaelis-Menten kinetics and input-output relations.
- Swain, PSB notes (enzyme kinetics section). [notes.pdf](https://swainlab.bio.ed.ac.uk/psb/lectures/notes.pdf) · [course site](https://swainlab.bio.ed.ac.uk/psb/index.html)
